# Task 1: Universal DAE Validation & Benchmark Evaluation

This notebook evaluates the final trained Task 1 Universal DAE model across all corruption types and severities on the deterministic test benchmark.

### Mathematical Formulation & Evaluation Metrics
The model reconstructs clean estimates from corrupted inputs:
$$\hat{x} = D\big(E(\tilde{x})\big)$$
Quantitative evaluation is performed against the ground-truth clean image $x$ using:
- **Pixel Reconstruction (L1)**: $\mathcal{L}_{L1} = \frac{1}{CHW} \|x - \hat{x}\|_1$
- **Structural Similarity (SSIM)**: preserving luminance, contrast, and structure
- **Peak Signal-to-Noise Ratio (PSNR)**:
  $$\mathrm{PSNR} = 10 \log_{10}\left(\frac{\mathrm{MAX}_I^2}{\mathrm{MSE}}\right) = 20 \log_{10}\left(\frac{1}{\sqrt{\mathrm{MSE}}}\right)$$
- **Absolute Error Map**: $|x - \hat{x}|$ highlighting localized restoration performance.

### Reporting Breakdown:
1. Independent results for **Clean**, **Salt-and-Pepper**, **Gaussian Blur**, and **Rectangular Occlusion**.
2. Independent results separated by **Low**, **Medium**, and **High** severities.
3. 12 representative examples and 4 failure cases with absolute error maps.
4. Latent space analysis (active dimensions, PCA, t-SNE, latent linear interpolation).

In [ ]:
# Imports and Evaluation Helpers
import sys
from pathlib import Path
from collections import defaultdict
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))
from src.device_utils import get_device
from src.models_dae import ConvDAE
from src.checkpoint_utils import load_checkpoint
from src.datasets import PetDataset
from src.evaluation import PetBenchmarkDataset, image_quality_metrics

DEVICE = get_device()


In [ ]:
# Configuration
FINAL_CHECKPOINT = RESEARCH_ROOT / "checkpoints" / "task1_universal_ae_final_best.pt"
BASELINE_CHECKPOINT = RESEARCH_ROOT / "checkpoints" / "task1_universal_ae_best.pt"
CHECKPOINT_PATH = FINAL_CHECKPOINT if FINAL_CHECKPOINT.exists() else BASELINE_CHECKPOINT
BASE_CHANNELS = 64
LATENT_DIM = 128
DROPOUT = 0.1


## 1. Model Loading

In [ ]:
# 1. Load the best trained model
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Train Task 1 first; missing checkpoint: {CHECKPOINT_PATH}")
saved = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
state = saved["model_state_dict"]
BASE_CHANNELS = state["enc1.0.weight"].shape[0]
LATENT_DIM = state["fc_z.weight"].shape[0]
DROPOUT = float(saved.get("dropout", DROPOUT))
model = ConvDAE(base_channels=BASE_CHANNELS, latent_dim=LATENT_DIM, dropout=DROPOUT).to(DEVICE)
checkpoint = load_checkpoint(CHECKPOINT_PATH, model, device=DEVICE)
model.eval()
print(f"Loaded epoch {checkpoint.get('epoch')} with validation loss {checkpoint.get('val_loss'):.4f}")


## 2. Quantitative Evaluation: PSNR, SSIM, and MSE Grid (Plan 6 §1.7)

In [ ]:
# 2. Fixed test benchmark: corruption x severity
benchmark = PetBenchmarkDataset(RESEARCH_ROOT / "test_manifest_official.json")
loader = DataLoader(benchmark, batch_size=16, shuffle=False, num_workers=2)
metric_sums = defaultdict(lambda: defaultdict(float))
metric_counts = defaultdict(int)
benchmark_records = []
extra_examples = []
failure_records = []
with torch.inference_mode():
    for corrupted, clean, labels, kinds, severities, names in loader:
        corrupted, clean = corrupted.to(DEVICE), clean.to(DEVICE)
        output = model(corrupted).clamp(0, 1)
        for i in range(len(clean)):
            metrics = image_quality_metrics(output[i:i+1], clean[i:i+1])
            key = (kinds[i], severities[i])
            metric_counts[key] += 1
            for metric, value in metrics.items():
                metric_sums[key][metric] += value
            record = {
                "input": corrupted[i].cpu(), "target": clean[i].cpu(),
                "output": output[i].cpu(), "kind": kinds[i],
                "severity": severities[i], "name": names[i], "metrics": metrics,
            }
            if not any(r["kind"] == kinds[i] and r["severity"] == severities[i] for r in benchmark_records):
                benchmark_records.append(record)
            elif len(extra_examples) < 2:
                extra_examples.append(record)
            failure_records.append(record)
            failure_records.sort(key=lambda r: r["metrics"]["psnr"])
            if len(failure_records) > 4:
                failure_records.pop()
rows = [{"corruption": kind, "severity": severity, "n": metric_counts[(kind, severity)],
         **{m: total / metric_counts[(kind, severity)] for m, total in metric_sums[(kind, severity)].items()}}
        for kind, severity in sorted(metric_counts)]
results_df = pd.DataFrame(rows)
display(results_df)


## 3. Visual Restorations & Absolute Error Maps

In [ ]:
# 3. Representative examples (target, input, output, absolute error)
chosen = []
for kind, severity in sorted({(r["kind"], r["severity"]) for r in benchmark_records}):
    candidates = [r for r in benchmark_records if r["kind"] == kind and r["severity"] == severity]
    chosen.append(candidates[len(candidates) // 2])
chosen = (chosen + extra_examples)[:12]
fig, axes = plt.subplots(len(chosen), 4, figsize=(13, 2.7 * len(chosen)), squeeze=False)
for row, record in zip(axes, chosen):
    images = [record["target"], record["input"], record["output"],
              (record["target"] - record["output"]).abs().mean(0)]
    for col, (ax, image, title) in enumerate(zip(row, images, ["Target", "Input", "Output", "Abs. error"])):
        ax.imshow(image.permute(1, 2, 0).numpy() if col < 3 else image.numpy(),
                  cmap=None if col < 3 else "inferno", vmin=0, vmax=0.5 if col == 3 else 1)
        ax.set_title(f"{record['kind']} / {record['severity']}\n{title}")
        ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Latent Space Analysis

In [ ]:
# 4. Latent representation: utilization, PCA/t-SNE, and interpolation
latent_vectors, latent_labels = [], []
val_loader = DataLoader(PetDataset(mode="val", corruption_mode="all", seed=42),
                        batch_size=32, shuffle=False, num_workers=2)
with torch.inference_mode():
    for corrupted, clean in val_loader:
        z, _ = model.encode(corrupted.to(DEVICE))
        latent_vectors.append(z.cpu())
latent = torch.cat(latent_vectors).numpy()
print(f"Latent vector std (mean across dims): {latent.std(axis=0).mean():.4f}; "
      f"active dimensions (std > 1e-3): {(latent.std(axis=0) > 1e-3).sum()}/{latent.shape[1]}")
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
sample = latent[:min(500, len(latent))]
pca = PCA(n_components=2, random_state=42).fit_transform(sample)
tsne = TSNE(n_components=2, init="pca", learning_rate="auto", random_state=42,
            perplexity=min(30, max(2, (len(sample) - 1) // 3))).fit_transform(sample)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(pca[:, 0], pca[:, 1], s=8, alpha=0.6); axes[0].set_title("Latent PCA")
axes[1].scatter(tsne[:, 0], tsne[:, 1], s=8, alpha=0.6); axes[1].set_title("Latent t-SNE")
plt.tight_layout(); plt.show()
z0, _ = model.encode(val_loader.dataset[0][0][None].to(DEVICE))
z1, _ = model.encode(val_loader.dataset[1][0][None].to(DEVICE))
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
with torch.inference_mode():
    for i, t in enumerate(torch.linspace(0, 1, 5)):
        decoded = model.decode(torch.lerp(z0, z1, t)).squeeze(0).cpu().permute(1, 2, 0).numpy()
        axes[i].imshow(decoded.clip(0, 1)); axes[i].set_title(f"t={t:.2f}"); axes[i].axis("off")
plt.tight_layout(); plt.show()


## 5. Failure Mode Analysis

In [ ]:
# 5. Four highest-error cases and failure hypotheses
failures = failure_records
fig, axes = plt.subplots(len(failures), 3, figsize=(10, 3 * len(failures)), squeeze=False)
for row, record in zip(axes, failures):
    for ax, image, title in zip(row, [record["input"], record["output"], record["target"]],
                                ["Corrupted input", "Reconstruction", "Target"]):
        ax.imshow(image.permute(1, 2, 0).numpy().clip(0, 1)); ax.set_title(title); ax.axis("off")
    print(f"{record['name']} ({record['kind']}/{record['severity']}): "
          f"PSNR={record['metrics']['psnr']:.2f} dB, SSIM={record['metrics']['ssim']:.3f}. "
          "Inspect whether information was removed by occlusion, overwhelmed by noise, or softened by blur.")
plt.tight_layout(); plt.show()
